# Perbandingan NER Legacy vs HuggingFace (400 News 2025)

Notebook ini membandingkan output NER model lama (base + `ner_model.sav`) dengan model baru yang sudah di-upload ke HuggingFace private repo.

Tujuan:
1. Memastikan output NER tidak berubah (exact match per row).
2. Mengecek apakah migrasi ke model HuggingFace mengatasi duplikasi artefak awal (estimasi pengurangan ukuran khusus jalur NER).


In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import BertConfig, BertTokenizer
from huggingface_hub import login

try:
    from dotenv import dotenv_values
except Exception:
    dotenv_values = None


In [ ]:
PROJECT_ROOT = Path.cwd()
TOOLKIT_ROOT = PROJECT_ROOT / "migration" / "ner_unification"
DATA_PATH = PROJECT_ROOT / "data" / "400_news_2025.csv"
LEGACY_CHECKPOINT = TOOLKIT_ROOT / "artifacts" / "ner_model.sav"
LABELS_PATH = TOOLKIT_ROOT / "config" / "ner_labels.json"
OUTPUT_DIR = TOOLKIT_ROOT / "output"
NER_HF_DIR = OUTPUT_DIR / "ner_hf"
REVISION_FILE = OUTPUT_DIR / "hf_revision.txt"

COMPARE_CSV = OUTPUT_DIR / "comparison_400_news_legacy_vs_hf.csv"
COMPARE_JSON = OUTPUT_DIR / "comparison_400_news_legacy_vs_hf.json"
SUMMARY_JSON = OUTPUT_DIR / "comparison_400_news_summary.json"

assert DATA_PATH.exists(), f"Data file tidak ditemukan: {DATA_PATH}"
assert LEGACY_CHECKPOINT.exists(), f"Checkpoint legacy tidak ditemukan: {LEGACY_CHECKPOINT}"
assert LABELS_PATH.exists(), f"Label config tidak ditemukan: {LABELS_PATH}"
assert REVISION_FILE.exists(), f"Revision file tidak ditemukan: {REVISION_FILE}"

if str(TOOLKIT_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOLKIT_ROOT))

from src.ner_modeling import (
    MODPURPOSE,
    build_label_maps,
    extract_state_dict,
    load_idx2tag,
    sanitize_state_dict_for_model,
)
from src.ner_inference import predict_entities

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)


In [ ]:
BASE_MODEL = "indolem/indobert-base-uncased"
HF_ENV_PATH = TOOLKIT_ROOT / ".env"
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO_ID = "AzrilFahmiardi/indobert-ner-prod-v1"

if HF_ENV_PATH.exists() and dotenv_values is not None:
    env = dotenv_values(HF_ENV_PATH)
    HF_TOKEN = env.get("HF_TOKEN") or HF_TOKEN
    HF_REPO_ID = env.get("HF_REPO_ID") or HF_REPO_ID

HF_REVISION = REVISION_FILE.read_text(encoding="utf-8").strip()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

print("HF_REPO_ID:", HF_REPO_ID)
print("HF_REVISION:", HF_REVISION)
print("HF_TOKEN set:", bool(HF_TOKEN))


In [ ]:
idx2tag = load_idx2tag(LABELS_PATH)
id2label, label2id = build_label_maps(idx2tag)

# Load legacy model (base + checkpoint .sav)
legacy_config = BertConfig.from_pretrained(BASE_MODEL)
legacy_config.num_labels = len(idx2tag)
legacy_config.id2label = {i: label for i, label in id2label.items()}
legacy_config.label2id = label2id

legacy_model = MODPURPOSE(legacy_config)
legacy_payload = torch.load(LEGACY_CHECKPOINT, map_location="cpu")
legacy_state = sanitize_state_dict_for_model(legacy_model, extract_state_dict(legacy_payload))
legacy_model.load_state_dict(legacy_state, strict=True)
legacy_tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)
legacy_model.eval()

# Load new model from HuggingFace private repo
hf_kwargs = {"revision": HF_REVISION} if HF_REVISION else {}
hf_config = BertConfig.from_pretrained(HF_REPO_ID, **hf_kwargs)
hf_model = MODPURPOSE.from_pretrained(HF_REPO_ID, config=hf_config, **hf_kwargs)
hf_tokenizer = BertTokenizer.from_pretrained(HF_REPO_ID, **hf_kwargs)
hf_model.eval()

print("Legacy + HF models loaded.")


In [ ]:
df = pd.read_csv(DATA_PATH)
if "summary" not in df.columns:
    raise ValueError("Kolom 'summary' tidak ditemukan di data/400_news_2025.csv")

df = df[df["summary"].notna()].copy()
df["summary"] = df["summary"].astype(str)
df = df.reset_index(drop=True)

print("Rows to compare:", len(df))
df.head(2)


In [ ]:
records = []
start = time.time()

for i, text in enumerate(df["summary"]):
    legacy_pred = predict_entities(text, legacy_model, legacy_tokenizer, idx2tag)
    hf_pred = predict_entities(text, hf_model, hf_tokenizer, idx2tag)
    records.append(
        {
            "row_index": int(i),
            "same": legacy_pred == hf_pred,
            "legacy": legacy_pred,
            "hf": hf_pred,
        }
    )

elapsed = time.time() - start

same_rows = sum(1 for r in records if r["same"])
different_rows = len(records) - same_rows

summary = {
    "total_rows": len(records),
    "same_rows": same_rows,
    "different_rows": different_rows,
    "match_rate": (same_rows / len(records)) if records else 0.0,
    "elapsed_seconds": elapsed,
}

summary


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame(
    [{"row_index": r["row_index"], "same": r["same"]} for r in records]
).to_csv(COMPARE_CSV, index=False)

with COMPARE_JSON.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False)

with SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

mismatches = [r for r in records if not r["same"]]
print("Summary:", summary)
print("Saved:", COMPARE_CSV)
print("Saved:", COMPARE_JSON)
print("Saved:", SUMMARY_JSON)

if mismatches:
    first = mismatches[0]
    print("First mismatch row_index:", first["row_index"])
    print("legacy:", json.dumps(first["legacy"], ensure_ascii=False))
    print("hf    :", json.dumps(first["hf"], ensure_ascii=False))
else:
    print("Semua output sama (exact match) untuk seluruh baris.")


In [ ]:
# Cek dampak terhadap masalah duplikasi ukuran (khusus jalur NER)
# Asumsi lama (dari audit produksi):
# - Base IndoBERT cache (blob): 444,780,374 bytes
# - Legacy checkpoint NER: ner_model.sav (lokal)
# Skema baru:
# - Unified model dari HF: model.safetensors

legacy_indobert_base_blob_size = 444_780_374
legacy_checkpoint_size = LEGACY_CHECKPOINT.stat().st_size
hf_model_weights_size = (NER_HF_DIR / "model.safetensors").stat().st_size

legacy_total_ner = legacy_indobert_base_blob_size + legacy_checkpoint_size
new_total_ner = hf_model_weights_size
saved_bytes = legacy_total_ner - new_total_ner
saved_pct = (saved_bytes / legacy_total_ner) if legacy_total_ner else 0.0

size_report = {
    "legacy_base_blob_size_bytes": legacy_indobert_base_blob_size,
    "legacy_checkpoint_size_bytes": legacy_checkpoint_size,
    "legacy_total_ner_bytes": legacy_total_ner,
    "new_hf_model_weights_bytes": hf_model_weights_size,
    "estimated_saved_bytes": saved_bytes,
    "estimated_saved_percent": saved_pct,
    "duplication_resolved_for_ner": saved_bytes > 0,
}

size_report


## Interpretasi

- Jika `different_rows = 0`, maka model baru di HuggingFace menghasilkan output NER yang sama dengan model legacy pada dataset ini.
- Jika `duplication_resolved_for_ner = True`, maka migrasi unified model **mengatasi duplikasi artefak NER** (base model + checkpoint terpisah) dan memberi estimasi pengurangan ukuran untuk jalur NER.
